# 图书库 SPARQL 练习 / Library SPARQL Practice

先运行本目录的 **start-fuseki.bat**，再选择 **Python (ARU)** 内核，从上到下运行。
Run **start-fuseki.bat** in this folder, select the **Python (ARU)** kernel and run all cells in order.

Python 通过 HTTP 向 Fuseki 的 `library` 数据集发送真实查询和修改请求。
Python sends real HTTP query and update requests to the Fuseki `library` dataset.

包含老师正文实际列出的 19 个示例和 12 道练习。每题可直接修改 `query` 中的 SPARQL 后重跑。
Includes the 19 examples present in the tutorial and all 12 homework tasks. Edit the SPARQL in each `query` and rerun its cell.

数据库保存在本目录的 `data/fuseki/library`，与 Notebook、图书原始数据一起迁移。
The persistent database is stored in this folder under `data/fuseki/library`, alongside the notebooks and source data.


## 1. 连接 Fuseki / Connect to Fuseki

两个函数分别负责查询和修改；仅向空的 `library` 数据集导入本目录的 `library.ttl`。
The two functions handle queries and updates. The local `library.ttl` is imported only when the `library` dataset is empty.

默认数据集地址为 `http://127.0.0.1:3030/library`。使用其他端口时，可修改下方 `DATASET`，或在启动 Jupyter 前设置 `FUSEKI_DATASET`。
The default dataset URL is `http://127.0.0.1:3030/library`. For another port, edit `DATASET` below or set `FUSEKI_DATASET` before starting Jupyter.


In [1]:
from pathlib import Path
from html import escape
import os
from urllib.error import URLError
from urllib.request import Request, urlopen
from IPython.display import display, HTML
from SPARQLWrapper import SPARQLWrapper, JSON, TURTLE, POST
from SPARQLWrapper.SPARQLExceptions import SPARQLWrapperException

# 查询和修改使用不同地址；可在别的端口运行 / Query and update use separate endpoints; another port is supported.
DATASET = os.environ.get('FUSEKI_DATASET', 'http://127.0.0.1:3030/library').rstrip('/')
answers = {}


def run_query(sparql, show=True):
    # 把 SPARQL 发给 Fuseki 执行 / Send SPARQL to Fuseki for execution.
    client = SPARQLWrapper(DATASET + '/query')
    client.setMethod(POST)
    client.setTimeout(20)
    client.setQuery(sparql)
    graph_result = client.queryType in ('CONSTRUCT', 'DESCRIBE')
    client.setReturnFormat(TURTLE if graph_result else JSON)
    result = client.query().convert()
    if show:
        if graph_result:
            print(result.decode('utf-8'))
        elif 'boolean' in result:
            print('是否匹配 / Match found:', '是 / True' if result['boolean'] else '否 / False')
        else:
            labels = {
                'title': '书名 / Title', 'author': '作者 / Author', 'authorName': '作者姓名 / Author name',
                'price': '价格 / Price', 'year': '年份 / Year', 'publisher': '出版社 / Publisher',
                'publisherName': '出版社名称 / Publisher name', 'name': '名称 / Name',
                'count': '数量 / Count', 'country': '国家 / Country', 'category': '价格档次 / Price category',
                'book': '书籍 / Book', 'subject': '主语 / Subject', 'property': '属性 / Property',
                'value': '值 / Value', 'person': '人物 / Person', 'fullName': '完整姓名 / Full name',
                'knownPerson': '认识的人 / Known person',
            }
            columns = result['head']['vars']
            header = ''.join('<th>' + escape(labels.get(v, v)) + '</th>' for v in columns)
            body = ''.join('<tr>' + ''.join('<td>' + escape(row.get(v, {}).get('value', '未记录 / Unbound'))
                           + '</td>' for v in columns) + '</tr>' for row in result['results']['bindings'])
            display(HTML('<table><tr>' + header + '</tr>' + body + '</table>'))
    return result


def run_update(sparql):
    # 修改也发送到同一个 library 数据集 / Send updates to the same library dataset.
    client = SPARQLWrapper(DATASET + '/update')
    client.setMethod(POST)
    client.setTimeout(20)
    client.setQuery(sparql)
    client.query().response.read()


# 用相对路径找作业数据，整个文件夹可以搬走 / Locate the coursework data using relative paths for portability.
candidates = [Path.cwd(), Path.cwd() / 'assignments' / 'Week3_NLP_SPARQL']
base = next((p for p in candidates if (p / 'library.ttl').is_file()), None)
if base is None:
    raise FileNotFoundError('请从作业目录或仓库根目录运行 / Run from the coursework folder or repository root.')

try:
    count = run_query('SELECT (COUNT(*) AS ?count) WHERE { ?s ?p ?o }', show=False)
    if int(count['results']['bindings'][0]['count']['value']) == 0:
        # 仅向空库导入，不覆盖已有数据 / Import only into an empty dataset; keep existing data intact.
        request = Request(DATASET + '/data?default', data=(base / 'library.ttl').read_bytes(),
                          headers={'Content-Type': 'text/turtle; charset=utf-8'}, method='POST')
        with urlopen(request, timeout=20) as response:
            response.read()
    print('已连接 Fuseki 图书库 / Connected to the Fuseki library dataset:', DATASET)
    connected = run_query('SELECT (COUNT(*) AS ?count) WHERE { ?s ?p ?o }')
except (URLError, SPARQLWrapperException) as error:
    raise RuntimeError('请先运行本作业的 start-fuseki.bat / Run start-fuseki.bat in this coursework folder first.') from error


已连接 Fuseki 图书库 / Connected to the Fuseki library dataset: http://127.0.0.1:3032/library


数量 / Count
42


## 2. 教程示例（Tutorial examples）

### 示例 01：查看三元组（Inspect triples）

主语、属性、值组成一条事实；LIMIT 最多返回 12 行。
A subject, property and value form one fact; LIMIT returns at most 12 rows.

In [2]:
# 主语、属性、值组成一条事实；LIMIT 最多返回 12 行。
# A subject, property and value form one fact; LIMIT returns at most 12 rows.
query = """
# 查看三元组 / Inspect triples
PREFIX ex: <http://example.org/library/>

SELECT ?subject ?property ?value
WHERE {
  ?subject ?property ?value .
}
LIMIT 12
"""
answers['Example01'] = run_query(query)

主语 / Subject,属性 / Property,值 / Value
http://example.org/library/jane,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/library/Author
http://example.org/library/jane,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/library/Person
http://example.org/library/jane,http://xmlns.com/foaf/0.1/name,Jane Austen
http://example.org/library/jane,http://example.org/library/fname,Jane
http://example.org/library/jane,http://example.org/library/lname,Austen
http://example.org/library/jane,http://xmlns.com/foaf/0.1/knows,http://example.org/library/charles
http://example.org/library/charles,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/library/Author
http://example.org/library/charles,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/library/Person
http://example.org/library/charles,http://xmlns.com/foaf/0.1/name,Charles Dickens
http://example.org/library/charles,http://example.org/library/fname,Charles


### 示例 02：查所有书籍（Find all books）

a 表示 rdf:type，也就是“属于某个类”。
The keyword a means rdf:type: the resource belongs to a class.

In [3]:
# a 表示 rdf:type，也就是“属于某个类”。
# The keyword a means rdf:type: the resource belongs to a class.
query = """
# 查所有书籍 / Find all books
PREFIX ex: <http://example.org/library/>

SELECT ?book
WHERE {
  ?book a ex:Book .
}
ORDER BY ?book
"""
answers['Example02'] = run_query(query)

书籍 / Book
http://example.org/library/book1
http://example.org/library/book2
http://example.org/library/book3
http://example.org/library/book4


### 示例 03：查所有书名（Find book titles）

WHERE 决定匹配哪些数据，SELECT 决定显示哪些列。
WHERE selects matching data; SELECT chooses the output columns.

In [4]:
# WHERE 决定匹配哪些数据，SELECT 决定显示哪些列。
# WHERE selects matching data; SELECT chooses the output columns.
query = """
# 查所有书名 / Find book titles
PREFIX ex: <http://example.org/library/>

SELECT ?title
WHERE {
  ?book a ex:Book ;
    ex:title ?title .
}
ORDER BY ?title
"""
answers['Example03'] = run_query(query)

书名 / Title
Emma
Mansfield Park
Oliver Twist
Pride and Prejudice


### 示例 04：查 Jane Austen 的作品（Find books by Jane Austen）

ex:jane 是作者的资源标识，不是姓名字符串。
ex:jane identifies the author resource; it is different from a name string.

In [5]:
# ex:jane 是作者的资源标识，不是姓名字符串。
# ex:jane identifies the author resource; it is different from a name string.
query = """
# 查 Jane Austen 的作品 / Find books by Jane Austen
PREFIX ex: <http://example.org/library/>

SELECT ?title
WHERE {
  ?book ex:author ex:jane ;
    ex:title ?title .
}
ORDER BY ?title
"""
answers['Example04'] = run_query(query)

书名 / Title
Emma
Mansfield Park
Pride and Prejudice


### 示例 05：连接作者姓名（Join author names）

共享变量 ?author 把书籍和作者姓名连接起来。
The shared variable ?author joins each book to its author name.

In [6]:
# 共享变量 ?author 把书籍和作者姓名连接起来。
# The shared variable ?author joins each book to its author name.
query = """
# 连接作者姓名 / Join author names
PREFIX ex: <http://example.org/library/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?title ?authorName
WHERE {
  ?book ex:title ?title ;
    ex:author ?author .
  ?author foaf:name ?authorName .
}
ORDER BY ?title
"""
answers['Example05'] = run_query(query)

书名 / Title,作者姓名 / Author name
Emma,Jane Austen
Mansfield Park,Jane Austen
Oliver Twist,Charles Dickens
Pride and Prejudice,Jane Austen


### 示例 06：筛选价格低于 50 的书（Filter prices below 50）

FILTER 保留满足价格条件的行。
FILTER keeps rows that satisfy the price condition.

In [7]:
# FILTER 保留满足价格条件的行。
# FILTER keeps rows that satisfy the price condition.
query = """
# 筛选价格低于 50 的书 / Filter prices below 50
PREFIX ex: <http://example.org/library/>

SELECT ?title ?price
WHERE {
  ?book ex:title ?title ;
    ex:price ?price .
  FILTER(?price < 50)
}
ORDER BY ?price
"""
answers['Example06'] = run_query(query)

书名 / Title,价格 / Price
Emma,25
Pride and Prejudice,45


### 示例 07：组合两个价格条件（Combine price conditions）

&& 表示两个条件都必须满足。
Both conditions joined by && must be true.

In [8]:
# && 表示两个条件都必须满足。
# Both conditions joined by && must be true.
query = """
# 组合两个价格条件 / Combine price conditions
PREFIX ex: <http://example.org/library/>

SELECT ?title ?price
WHERE {
  ?book ex:title ?title ;
    ex:price ?price .
  FILTER(?price > 30 && ?price < 60)
}
ORDER BY ?price
"""
answers['Example07'] = run_query(query)

书名 / Title,价格 / Price
Pride and Prejudice,45
Mansfield Park,55


### 示例 08：保留缺少年份的书（Include missing years）

OPTIONAL 表示可选信息；查不到年份时仍保留书名。
OPTIONAL keeps the title even when the publication year is missing.

In [9]:
# OPTIONAL 表示可选信息；查不到年份时仍保留书名。
# OPTIONAL keeps the title even when the publication year is missing.
query = """
# 保留缺少年份的书 / Include missing years
PREFIX ex: <http://example.org/library/>

SELECT ?title ?year
WHERE {
  ?book ex:title ?title .
  OPTIONAL {
  ?book ex:year ?year .
}
}
ORDER BY ?title
"""
answers['Example08'] = run_query(query)

书名 / Title,年份 / Year
Emma,1815
Mansfield Park,未记录 / Unbound
Oliver Twist,1838
Pride and Prejudice,1813


### 示例 09：查没有记录年份的书（Find books without a recorded year）

BOUND 检查变量是否绑定；!BOUND 表示没有查到值。
BOUND checks whether a variable has a value; !BOUND selects missing values.

In [10]:
# BOUND 检查变量是否绑定；!BOUND 表示没有查到值。
# BOUND checks whether a variable has a value; !BOUND selects missing values.
query = """
# 查没有记录年份的书 / Find books without a recorded year
PREFIX ex: <http://example.org/library/>

SELECT ?title
WHERE {
  ?book ex:title ?title .
  OPTIONAL {
  ?book ex:year ?year .
} FILTER(!BOUND(?year))
}
"""
answers['Example09'] = run_query(query)

书名 / Title
Mansfield Park


### 示例 10：用 UNION 合并两类结果（Combine alternatives with UNION）

两个图模式满足其中一个就可以。
A match in either graph pattern is sufficient.

In [11]:
# 两个图模式满足其中一个就可以。
# A match in either graph pattern is sufficient.
query = """
# 用 UNION 合并两类结果 / Combine alternatives with UNION
PREFIX ex: <http://example.org/library/>

SELECT ?title
WHERE {
  ?book ex:title ?title .
  {
  ?book ex:author ex:jane .
} UNION {
  ?book ex:author ex:charles .
}
}
ORDER BY ?title
"""
answers['Example10'] = run_query(query)

书名 / Title
Emma
Mansfield Park
Oliver Twist
Pride and Prejudice


### 示例 11：用 MINUS 排除结果（Exclude matches with MINUS）

排除与 Charles Dickens 关联的书。
Exclude books linked to Charles Dickens.

In [12]:
# 排除与 Charles Dickens 关联的书。
# Exclude books linked to Charles Dickens.
query = """
# 用 MINUS 排除结果 / Exclude matches with MINUS
PREFIX ex: <http://example.org/library/>

SELECT ?title
WHERE {
  ?book a ex:Book ;
    ex:title ?title .
  MINUS {
  ?book ex:author ex:charles .
}
}
ORDER BY ?title
"""
answers['Example11'] = run_query(query)

书名 / Title
Emma
Mansfield Park
Pride and Prejudice


### 示例 12：用 VALUES 指定候选书籍（Restrict choices with VALUES）

只从给出的两个书籍标识中匹配。
Match only the two supplied book identifiers.

In [13]:
# 只从给出的两个书籍标识中匹配。
# Match only the two supplied book identifiers.
query = """
# 用 VALUES 指定候选书籍 / Restrict choices with VALUES
PREFIX ex: <http://example.org/library/>

SELECT ?title
WHERE {
  VALUES ?book {
  ex:book1 ex:book4
} ?book ex:title ?title .
}
ORDER BY ?title
"""
answers['Example12'] = run_query(query)

书名 / Title
Oliver Twist
Pride and Prejudice


### 示例 13：用 BIND 拼接完整姓名（Build full names with BIND）

CONCAT 拼接名字；BIND 得到临时查询变量，不修改数据库。
CONCAT joins name strings; BIND creates a query variable without changing stored data.

In [14]:
# CONCAT 拼接名字；BIND 得到临时查询变量，不修改数据库。
# CONCAT joins name strings; BIND creates a query variable without changing stored data.
query = """
# 用 BIND 拼接完整姓名 / Build full names with BIND
PREFIX ex: <http://example.org/library/>

SELECT ?person ?fullName
WHERE {
  ?person a ex:Author ;
    ex:fname ?first ;
    ex:lname ?last .
  BIND(CONCAT(?first, " ", ?last) AS ?fullName)
}
ORDER BY ?fullName
"""
answers['Example13'] = run_query(query)

人物 / Person,完整姓名 / Full name
http://example.org/library/charles,Charles Dickens
http://example.org/library/jane,Jane Austen


### 示例 14：用 DISTINCT 去重（Remove duplicates with DISTINCT）

即使多本书对应同一个国家，结果也只显示一次。
Return a country once even when several books are linked to it.

In [15]:
# 即使多本书对应同一个国家，结果也只显示一次。
# Return a country once even when several books are linked to it.
query = """
# 用 DISTINCT 去重 / Remove duplicates with DISTINCT
PREFIX ex: <http://example.org/library/>

SELECT DISTINCT ?country
WHERE {
  ?book ex:publisher/ex:locatedIn/ex:country ?country .
}
"""
answers['Example14'] = run_query(query)

国家 / Country
United Kingdom


### 示例 15：沿属性路径查询国家（Follow a property path）

/ 依次连接出版社、所在地和国家。
The slash follows publisher, location and country relationships in sequence.

In [16]:
# / 依次连接出版社、所在地和国家。
# The slash follows publisher, location and country relationships in sequence.
query = """
# 沿属性路径查询国家 / Follow a property path
PREFIX ex: <http://example.org/library/>

SELECT ?title ?country
WHERE {
  ?book ex:title ?title ;
    ex:publisher/ex:locatedIn/ex:country ?country .
}
"""
answers['Example15'] = run_query(query)

书名 / Title,国家 / Country
Pride and Prejudice,United Kingdom
Emma,United Kingdom
Mansfield Park,United Kingdom
Oliver Twist,United Kingdom


### 示例 16：用 ASK 判断是否存在（Check existence with ASK）

ASK 返回真或假，不返回结果表。
ASK returns a Boolean rather than a result table.

In [17]:
# ASK 返回真或假，不返回结果表。
# ASK returns a Boolean rather than a result table.
query = """
# 用 ASK 判断是否存在 / Check existence with ASK
PREFIX ex: <http://example.org/library/>

ASK
WHERE {
  ex:book2 ex:author ex:jane .
}
"""
answers['Example16'] = run_query(query)

是否匹配 / Match found: 是 / True


### 示例 17：按作者统计书数（Count books by author）

GROUP BY 先分组，COUNT 再统计每组数量。
GROUP BY forms groups; COUNT counts the books in each group.

In [18]:
# GROUP BY 先分组，COUNT 再统计每组数量。
# GROUP BY forms groups; COUNT counts the books in each group.
query = """
# 按作者统计书数 / Count books by author
PREFIX ex: <http://example.org/library/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?authorName (COUNT(?book) AS ?count)
WHERE {
  ?book a ex:Book ;
    ex:author ?author .
  ?author foaf:name ?authorName .
}
GROUP BY ?authorName
ORDER BY DESC(?count)
"""
answers['Example17'] = run_query(query)

作者姓名 / Author name,数量 / Count
Jane Austen,3
Charles Dickens,1


### 示例 18：用 CONSTRUCT 生成新图（Build a graph with CONSTRUCT）

返回新 RDF 图，本条查询不会向数据库写入。
Return a new RDF graph without inserting its triples into the database.

In [19]:
# 返回新 RDF 图，本条查询不会向数据库写入。
# Return a new RDF graph without inserting its triples into the database.
query = """
# 用 CONSTRUCT 生成新图 / Build a graph with CONSTRUCT
PREFIX ex: <http://example.org/library/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

CONSTRUCT {
  ?book ex:authorName ?name .
}
WHERE {
  ?book ex:author ?author .
  ?author foaf:name ?name .
}
"""
answers['Example18'] = run_query(query)

PREFIX ex:   <http://example.org/library/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>

ex:book4  ex:authorName  "Charles Dickens" .

ex:book3  ex:authorName  "Jane Austen" .

ex:book2  ex:authorName  "Jane Austen" .

ex:book1  ex:authorName  "Jane Austen" .



### 示例 19：用 DESCRIBE 描述一本书（Describe a book）

由 Fuseki 返回该资源的描述，范围可能与其他引擎不同。
Fuseki chooses the description; its scope may differ from other engines.

In [20]:
# 由 Fuseki 返回该资源的描述，范围可能与其他引擎不同。
# Fuseki chooses the description; its scope may differ from other engines.
query = """
# 用 DESCRIBE 描述一本书 / Describe a book
PREFIX ex: <http://example.org/library/>

DESCRIBE ex:book1
"""
answers['Example19'] = run_query(query)

PREFIX ex:   <http://example.org/library/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>

ex:book1  rdf:type    ex:Book;
        ex:author     ex:jane;
        ex:price      45;
        ex:publisher  ex:pressA;
        ex:title      "Pride and Prejudice"@en;
        ex:year       1813 .



## 3. 十二道作业（Homework）

### 练习 1：查出版社及名称（Find publishers and names）

查出出版社资源以及它对应的名称。
Return each publisher resource and its name.

In [21]:
# 查出出版社资源以及它对应的名称。
# Return each publisher resource and its name.
query = """
# 查出版社及名称 / Find publishers and names
PREFIX ex: <http://example.org/library/>

SELECT ?publisher ?name
WHERE {
  ?publisher a ex:Publisher ;
    ex:name ?name .
}
ORDER BY ?publisher
"""
answers['Homework01'] = run_query(query)

出版社 / Publisher,名称 / Name
http://example.org/library/pressA,Classics Press
http://example.org/library/pressB,Heritage Books


### 练习 2：价格至少 50，按价格降序（Sort prices of at least 50）

>= 表示大于或等于；DESC 表示降序。
The operator >= includes the boundary; DESC sorts in descending order.

In [22]:
# >= 表示大于或等于；DESC 表示降序。
# The operator >= includes the boundary; DESC sorts in descending order.
query = """
# 价格至少 50，按价格降序 / Sort prices of at least 50
PREFIX ex: <http://example.org/library/>

SELECT ?title ?price
WHERE {
  ?book ex:title ?title ;
    ex:price ?price .
  FILTER(?price >= 50)
}
ORDER BY DESC(?price)
"""
answers['Homework02'] = run_query(query)

书名 / Title,价格 / Price
Oliver Twist,70
Mansfield Park,55


### 练习 3：按书名字母顺序取两本（Return two titles alphabetically）

ORDER BY 按书名排序，LIMIT 2 只取前两条。
ORDER BY sorts the titles; LIMIT 2 returns only the first two.

In [23]:
# ORDER BY 按书名排序，LIMIT 2 只取前两条。
# ORDER BY sorts the titles; LIMIT 2 returns only the first two.
query = """
# 按书名字母顺序取两本 / Return two titles alphabetically
PREFIX ex: <http://example.org/library/>

SELECT ?title
WHERE {
  ?book a ex:Book ;
    ex:title ?title .
}
ORDER BY ?title
LIMIT 2
"""
answers['Homework03'] = run_query(query)

书名 / Title
Emma
Mansfield Park


### 练习 4：书名和出版社名称（Join titles and publisher names）

先从书找到出版社，再从出版社找到名称。
Follow a book to its publisher, then retrieve the publisher name.

In [24]:
# 先从书找到出版社，再从出版社找到名称。
# Follow a book to its publisher, then retrieve the publisher name.
query = """
# 书名和出版社名称 / Join titles and publisher names
PREFIX ex: <http://example.org/library/>

SELECT ?title ?publisherName
WHERE {
  ?book ex:title ?title ;
    ex:publisher ?publisher .
  ?publisher ex:name ?publisherName .
}
ORDER BY ?title
"""
answers['Homework04'] = run_query(query)

书名 / Title,出版社名称 / Publisher name
Emma,Heritage Books
Mansfield Park,Classics Press
Oliver Twist,Heritage Books
Pride and Prejudice,Classics Press


### 练习 5：用 NOT EXISTS 查缺少年份的书（Find missing years with NOT EXISTS）

不用 OPTIONAL；不存在年份事实时保留该书。
Keep a book when no year triple exists, without using OPTIONAL.

In [25]:
# 不用 OPTIONAL；不存在年份事实时保留该书。
# Keep a book when no year triple exists, without using OPTIONAL.
query = """
# 用 NOT EXISTS 查缺少年份的书 / Find missing years with NOT EXISTS
PREFIX ex: <http://example.org/library/>

SELECT ?title
WHERE {
  ?book a ex:Book ;
    ex:title ?title .
  FILTER NOT EXISTS {
  ?book ex:year ?year .
}
}
ORDER BY ?title
"""
answers['Homework05'] = run_query(query)

书名 / Title
Mansfield Park


### 练习 6：用 UNION 查两家出版社的书（Use UNION for two publishers）

返回 pressA 或 pressB 出版的书。
Return books published by either pressA or pressB.

In [26]:
# 返回 pressA 或 pressB 出版的书。
# Return books published by either pressA or pressB.
query = """
# 用 UNION 查两家出版社的书 / Use UNION for two publishers
PREFIX ex: <http://example.org/library/>

SELECT ?title
WHERE {
  ?book ex:title ?title .
  {
  ?book ex:publisher ex:pressA .
} UNION {
  ?book ex:publisher ex:pressB .
}
}
ORDER BY ?title
"""
answers['Homework06'] = run_query(query)

书名 / Title
Emma
Mansfield Park
Oliver Twist
Pride and Prejudice


### 练习 7：用 VALUES 选择价格 25 或 70（Choose prices with VALUES）

只匹配价格为 25 或 70 的书。
Match only books with a price of 25 or 70.

In [27]:
# 只匹配价格为 25 或 70 的书。
# Match only books with a price of 25 or 70.
query = """
# 用 VALUES 选择价格 25 或 70 / Choose prices with VALUES
PREFIX ex: <http://example.org/library/>

SELECT ?title ?price
WHERE {
  VALUES ?price {
  25 70
} ?book ex:title ?title ;
    ex:price ?price .
}
ORDER BY ?price
"""
answers['Homework07'] = run_query(query)

书名 / Title,价格 / Price
Emma,25
Oliver Twist,70


### 练习 8：用 IF 和 BIND 区分价格档次（Label prices with IF and BIND）

budget 表示经济档，premium 表示较高价格档。
Label prices below 50 as budget and all remaining prices as premium.

In [28]:
# budget 表示经济档，premium 表示较高价格档。
# Label prices below 50 as budget and all remaining prices as premium.
query = """
# 用 IF 和 BIND 区分价格档次 / Label prices with IF and BIND
PREFIX ex: <http://example.org/library/>

SELECT ?title ?price ?category
WHERE {
  ?book ex:title ?title ;
    ex:price ?price .
  BIND(IF(?price < 50, "budget", "premium") AS ?category)
}
ORDER BY ?title
"""
answers['Homework08'] = run_query(query)

书名 / Title,价格 / Price,价格档次 / Price category
Emma,25,budget
Mansfield Park,55,premium
Oliver Twist,70,premium
Pride and Prejudice,45,budget


### 练习 9：沿一个或多个认识关系查询（Follow one or more knows links）

foaf:knows+ 沿有向边前进；Jane 认识 Charles，不能据此推出反向关系。
foaf:knows+ follows directed edges; Jane knowing Charles does not imply the reverse relationship.

In [29]:
# foaf:knows+ 沿有向边前进；Jane 认识 Charles，不能据此推出反向关系。
# foaf:knows+ follows directed edges; Jane knowing Charles does not imply the reverse relationship.
query = """
# 沿一个或多个认识关系查询 / Follow one or more knows links
PREFIX ex: <http://example.org/library/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT DISTINCT ?author ?knownPerson
WHERE {
  ?author a ex:Author ;
    foaf:knows+ ?knownPerson .
}
ORDER BY ?author ?knownPerson
"""
answers['Homework09'] = run_query(query)

作者 / Author,认识的人 / Known person
http://example.org/library/jane,http://example.org/library/charles


### 练习 10：按出版社统计书数（Count books by publisher）

按出版社分组统计；书数相同时再按出版社名称排序。
Count books by publisher; use the publisher name to break ties.

In [30]:
# 按出版社分组统计；书数相同时再按出版社名称排序。
# Count books by publisher; use the publisher name to break ties.
query = """
# 按出版社统计书数 / Count books by publisher
PREFIX ex: <http://example.org/library/>

SELECT ?publisherName (COUNT(?book) AS ?count)
WHERE {
  ?book a ex:Book ;
    ex:publisher ?publisher .
  ?publisher ex:name ?publisherName .
}
GROUP BY ?publisherName
ORDER BY DESC(?count) ?publisherName
"""
answers['Homework10'] = run_query(query)

出版社名称 / Publisher name,数量 / Count
Classics Press,2
Heritage Books,2


### 练习 11：构造低价书推荐链接（Construct recommendation links）

ex:budgetList 表示推荐列表，ex:recommendedTitle 连接被推荐的书。
ex:budgetList is a recommendation list; ex:recommendedTitle links it to recommended books.

In [31]:
# ex:budgetList 表示推荐列表，ex:recommendedTitle 连接被推荐的书。
# ex:budgetList is a recommendation list; ex:recommendedTitle links it to recommended books.
query = """
# 构造低价书推荐链接 / Construct recommendation links
PREFIX ex: <http://example.org/library/>

CONSTRUCT {
  ex:budgetList ex:recommendedTitle ?book .
}
WHERE {
  ?book a ex:Book ;
    ex:price ?price .
  FILTER(?price < 50)
}
"""
answers['Homework11'] = run_query(query)

PREFIX ex:   <http://example.org/library/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>

ex:budgetList  ex:recommendedTitle  ex:book2 , ex:book1 .



### 练习 12：新增、修改、删除年份（Insert, change and remove a year）

观察同一条 SELECT 的四次结果：操作前为空；新增后年份是 2026；修改后是 2027；仅删除年份后，书仍存在但年份未记录。

本题只操作 `ex:practiceBook` 这本临时书，结束后清理它，方便从头重复练习。

Observe the same SELECT before changes, after inserting the book (2026), after changing its year (2027), and after removing only the year (unbound while the book remains).
Only the reserved temporary resource `ex:practiceBook` is used. It is cleaned up at the end so the exercise can be repeated.


In [32]:
# 反复使用同一条查询观察变化 / Use the same SELECT to observe each stage.
query_12 = """
PREFIX ex: <http://example.org/library/>

SELECT ?title ?author ?year
WHERE {
  ex:practiceBook ex:title ?title ;
    ex:author ?author .
  OPTIONAL {
  ex:practiceBook ex:year ?year .
}
}
"""

# 只清理上次练习的临时书 / Clear only the temporary book from a previous run.
cleanup = """
PREFIX ex: <http://example.org/library/>
DELETE WHERE { ex:practiceBook ?p ?o . }
"""
run_update(cleanup)
stages = []
try:
    print('操作之前 / Before changes')
    stages.append(run_query(query_12))

    # 新增之后 / After inserting the book.
    run_update("""
PREFIX ex: <http://example.org/library/>

INSERT DATA {
  ex:practiceBook a ex:Book ;
    ex:title "Practice Book"@en ;
    ex:author ex:jane ;
    ex:year 2026 .
}
""")
    print('新增之后 / After inserting the book')
    stages.append(run_query(query_12))

    # 修改年份之后 / After changing the year.
    run_update("""
PREFIX ex: <http://example.org/library/>

DELETE {
  ex:practiceBook ex:year ?old .
} INSERT {
  ex:practiceBook ex:year 2027 .
}
WHERE {
  ex:practiceBook ex:year ?old .
}
""")
    print('修改年份之后 / After changing the year')
    stages.append(run_query(query_12))

    # 仅删除年份之后 / After removing only the year.
    run_update("""
PREFIX ex: <http://example.org/library/>

DELETE DATA {
  ex:practiceBook ex:year 2027 .
}
""")
    print('仅删除年份之后 / After removing only the year')
    stages.append(run_query(query_12))

finally:
    # 保留课堂原始数据，移除临时书 / Keep the course data and remove the temporary book.
    run_update(cleanup)


操作之前 / Before changes


书名 / Title,作者 / Author,年份 / Year


新增之后 / After inserting the book


书名 / Title,作者 / Author,年份 / Year
Practice Book,http://example.org/library/jane,2026


修改年份之后 / After changing the year


书名 / Title,作者 / Author,年份 / Year
Practice Book,http://example.org/library/jane,2027


仅删除年份之后 / After removing only the year


书名 / Title,作者 / Author,年份 / Year
Practice Book,http://example.org/library/jane,未记录 / Unbound


## 4. 运行自检 / Execution checks

核对所有示例和作业的结果，并确认第 12 题清理后仍为原始 42 条三元组。更改练习数据时，预期结果也需要调整。
Check the results of every example and homework task, including the original 42 triples after task 12 has cleaned up. Update the expected results if you change the practice data.

In [33]:
from rdflib import Graph, Namespace, URIRef

# 对老师原始数据做自检，数据改动后需同步预期值 / Check the original course data; update expectations if the data changes.
expected_counts = [12, 4, 4, 3, 4, 2, 2, 4, 1, 4, 3, 2, 2, 1, 4, None, 2, 4, None]
for number, expected in enumerate(expected_counts, 1):
    result = answers[f'Example{number:02}']
    if number == 16:
        assert result['boolean'] is True
    elif number in (18, 19):
        returned_graph = Graph().parse(data=result, format='turtle')
        if number == 18:
            assert len(returned_graph) == expected
        else:
            original_graph = Graph().parse(base / 'library.ttl', format='turtle')
            book1 = URIRef('http://example.org/library/book1')
            assert set(original_graph.triples((book1, None, None))).issubset(set(returned_graph))
    else:
        assert len(result['results']['bindings']) == expected, f'Example{number:02}'

for number, expected in enumerate([2, 2, 2, 4, 1, 4, 2, 4, 1, 2, 2], 1):
    result = answers[f'Homework{number:02}']
    actual = len(Graph().parse(data=result, format='turtle')) if number == 11 else len(result['results']['bindings'])
    assert actual == expected, f'Homework{number:02}'

# 不只检查行数，也检查边界、价格、分组和四阶段修改 / Check values, price boundaries, groups and all four update stages.
def values(key, *columns):
    return [[row.get(column, {}).get('value') for column in columns] for row in answers[key]['results']['bindings']]

assert values('Example06', 'title', 'price') == [['Emma', '25'], ['Pride and Prejudice', '45']]
assert values('Example17', 'authorName', 'count') == [['Jane Austen', '3'], ['Charles Dickens', '1']]
assert values('Homework02', 'title', 'price') == [['Oliver Twist', '70'], ['Mansfield Park', '55']]
assert values('Homework03', 'title') == [['Emma'], ['Mansfield Park']]
assert values('Homework05', 'title') == [['Mansfield Park']]
assert values('Homework10', 'publisherName', 'count') == [['Classics Press', '2'], ['Heritage Books', '2']]
assert stages[0]['results']['bindings'] == []
assert [stage['results']['bindings'][0].get('year', {}).get('value') for stage in stages[1:]] == ['2026', '2027', None]
assert not run_query('PREFIX ex: <http://example.org/library/> ASK { ex:practiceBook ?p ?o }', show=False)['boolean']
remaining = run_query('SELECT (COUNT(*) AS ?count) WHERE { ?s ?p ?o }', show=False)
assert remaining['results']['bindings'][0]['count']['value'] == '42'
assert len(answers) == 30
print('通过：19 个示例、12 道作业；临时书已清理 / Passed: 19 examples and 12 homework tasks; temporary book removed.')


通过：19 个示例、12 道作业；临时书已清理 / Passed: 19 examples and 12 homework tasks; temporary book removed.
